In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.7 Serve via inference API

Finally, put the model behind a request/response handler. `InferenceService` is
transport-agnostic: the same `handle` method works in a test, a queue worker, or a
FastAPI route.

In [ ]:
from pocketlm import train_tiny, pick_config, InferenceService

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
svc = InferenceService(model, tok)
resp = svc.handle({"prompt": "The ", "max_new_tokens": 20, "seed": 0})
print(resp["completion"])

Wrapping it in FastAPI is a few lines (shown, not run here, to avoid a server in CI):

```python
from fastapi import FastAPI
app = FastAPI()
svc = InferenceService(model, tok)

@app.post("/generate")
def generate_route(req: dict):
    return svc.handle(req)
# uvicorn serves `app`; the handler is exactly what we tested above.
```

That is the whole course end to end: tokenize, build, train, modernize, adapt, and
now serve a decoder-only language model you wrote yourself.

In [ ]:
assert resp["prompt"] == "The "
assert len(resp["completion"]) == len("The ") + 20